In [84]:
import lpips
import torch
from PIL import Image
import cv2
import torchvision.transforms as transforms
import warnings
from skimage.metrics import structural_similarity as ssim
import os
from cleanfid import fid
warnings.filterwarnings("ignore", category=UserWarning)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
def calculate_fid(pred_dir, true_dir):
    fid_value = fid.compute_fid(pred_dir, true_dir,batch_size=4,num_workers=0,device=device,mode="clean")
    return fid_value
def calculate_lpips(image1_path, image2_path, net_type='alex'):
    loss_fn = lpips.LPIPS(net=net_type)
    loss_fn.to(device)
    loss_fn.eval()
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
    ])
    img1 = Image.open(image1_path).convert('RGB')
    img2 = Image.open(image2_path).convert('RGB')
    tensor1 = transform(img1).unsqueeze(0).to(device)
    tensor2 = transform(img2).unsqueeze(0).to(device)
    with torch.no_grad():
        lpips_value = loss_fn(tensor1, tensor2)
    return lpips_value.item()
def calculate_lpips_depth(image1_path, image2_path, net_type='alex'):
    loss_fn = lpips.LPIPS(net=net_type)
    loss_fn.to(device)
    loss_fn.eval()
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
    ])
    img1 = Image.open(image1_path)
    img2 = Image.open(image2_path)
    img1=np.array(img1)[:,:,0]
    img2=np.array(img2)
    img2 = cv2.resize(img2, img1.shape, 
                          interpolation=cv2.INTER_LINEAR).transpose(1,0)
    img1=np.stack([img1,img1,img1],axis=-1)
    
    img2=np.stack([img2,img2,img2],axis=-1)
    
    tensor1 = transform(img1).unsqueeze(0).to(device)
    tensor2 = transform(img2).unsqueeze(0).to(device)
    with torch.no_grad():
        lpips_value = loss_fn(tensor1, tensor2)
    return lpips_value.item()
def calculate_pixel_metrics(pred_path, true_path):
    pred = cv2.imread(pred_path).astype(np.float32)
    true = cv2.imread(true_path).astype(np.float32)[:,:,0]
    
    # PSNR
    mse = np.mean((pred - true) ** 2)
    psnr = 20 * np.log10(255.0 / np.sqrt(mse))
    
    # SSIM
    ssim_score = ssim(pred, true, win_size=7,channel_axis=2, data_range=255)
    
    return psnr, ssim_score
def calculate_pixel_metrics_depth(pred_path, true_path):
    pred = cv2.imread(pred_path).astype(np.float32)
    true = cv2.imread(true_path).astype(np.float32)
    pred=cv2.resize(pred,(true.shape[0],true.shape[1]),interpolation=cv2.INTER_LINEAR).transpose(1,0,2)
    # PSNR
    mse = np.mean((pred - true) ** 2)
    psnr = 20 * np.log10(255.0 / np.sqrt(mse))
    
    # SSIM
    ssim_score = ssim(pred, true, win_size=7,channel_axis=2, data_range=255)
    
    return psnr, ssim_score

In [2]:
imgs_root="assets/"

In [61]:
lpips_sum=0
for i in range(20):
    lpips_score = calculate_lpips_depth(imgs_root+f'depth_truth/depth{i}.png', imgs_root+f'depths_truth_right/depth_{i}.png')
    lpips_sum+=lpips_score
print(f"平均LPIPS:{(lpips_sum/20):.4f}")

Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]
Loading model from: D:\miniconda\envs\myenv\lib\site-packages\lpips\weights\v0.1\alex.pth
Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]
Loading model from: D:\miniconda\envs\myenv\lib\site-packages\lpips\weights\v0.1\alex.pth
Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]
Loading model from: D:\miniconda\envs\myenv\lib\site-packages\lpips\weights\v0.1\alex.pth
Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]
Loading model from: D:\miniconda\envs\myenv\lib\site-packages\lpips\weights\v0.1\alex.pth
Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]
Loading model from: D:\miniconda\envs\myenv\lib\site-packages\lpips\weights\v0.1\alex.pth
Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]
Loading model from: D:\miniconda\envs\myenv\lib\site-packages\lpips\weights\v0.1\alex.pth
Setting up [LPIPS] perceptua

In [86]:
psnr_sum=0
ssim_sum=0
for i in range(20):
    psnr,ssim_score=calculate_pixel_metrics_depth(imgs_root+f'depths_predict_right/depth_{i}.png',imgs_root+f'depth_truth/depth{i}.png')
    psnr_sum+=psnr
    ssim_sum+=ssim_score
print(f"平均psnr:{(psnr_sum/20):.4f}")
print(f"平均ssim:{(ssim_sum/20):.4f}")

平均psnr:4.9949
平均ssim:0.4923


In [ ]:
fid_value=calculate_fid(imgs_root+"rights",imgs_root+"rights_pre")
print(f"fid_value:{fid_value:.4f}")

In [62]:
truth=cv2.imread("assets/depth_truth/depth0.png").astype(np.float32)

In [75]:
truth.shape

(415, 906, 3)

In [67]:
pre=cv2.imread("assets/depths_predict_right/depth_0.png").astype(np.float32)

In [80]:
pre=cv2.resize(pre,(truth.shape[0],truth.shape[1]),interpolation=cv2.INTER_LINEAR).transpose(1,0,2)

In [81]:
pre.shape

(415, 906, 3)